# Disease heads v2 - full label inventory, grouped CV selection

Rebuilds the `multicrop_expanded_bundle` disease heads so that **every PlantVillage class the data actually
contains is servable**, instead of the 11-of-38 coverage of v1. Same frozen encoders and the same 2816-dim
feature layout, so `main.py` can serve the result unchanged.

## What v2 fixes (each one is a concrete v1 defect)

| v1 defect | v2 fix |
|---|---|
| Only 11 of 38 PlantVillage classes had a head (Apple/Corn/Grape 2 of 4, no Pepper_bell, no Orange) | heads are built from the **scanned label inventory**: every crop with >= 2 usable conditions gets a head, class set = observed classes |
| Peach / Cherry / Strawberry answered "healthy" although PlantVillage has disease folders for them | `single_class_crops` is now **derived from the data**, not hard-coded |
| Potato head had no healthy class | class set comes from the inventory, so healthy is included when present |
| `tomato_experiment='tv_pd'` silently swapped the 6-class Tomato head for the old 3-class one | the exported config has **no `tomato_experiment` override**; `disease_heads['Tomato']` is the 6+ class head |
| Epoch chosen on a tiny PlantDoc-only validation split (`best_epoch: 1`, flat F1) | epoch chosen by **5-fold grouped CV out-of-fold macro F1**; PlantDoc validation is only reported |
| Broad heads selected on `cap` sweep noise | one documented `PV_CAP_RATIO`, no cap sweep |
| `Apple rust leaf`, `Corn leaf blight`, `Bell pepper leaf spot` dropped from training | explicit `PD_MAPPING_OVERRIDES`, gated by a must-set `PD_MAPPING_OVERRIDES_VERIFIED` flag |
| Frozen features limited to CLIP embedding + DINO CLS + mean patch tokens | `FEATURE_MODE='topk_norm'`: mean of the **top-8 highest-norm DINO patches**, which keeps the lesion region instead of averaging it away |

## Before Run All

1. Enable a Kaggle **GPU**.
2. Attach PlantVillage, PlantDoc (classification, `train/` + `test/` folders) and the existing bundle
   (for `clip/` + `dino/` weights). Point the paths in Cell 2 at them.
3. Eyeball the three PlantDoc folders listed in `PD_MAPPING_OVERRIDES`, then set
   `PD_MAPPING_OVERRIDES_VERIFIED = True` (or clear the dict). The notebook refuses to guess silently.
4. Run All. Export is a zip; drop it in as the new bundle and restart the API.

## Scope / honesty

- Selection uses grouped CV out-of-fold macro F1 over PlantVillage + PlantDoc training rows. The PlantDoc
  validation rows are reported but not used to pick epochs, and the PlantDoc test rows are never touched
  during training.
- Duplicate groups are exact byte duplicates plus an optional shared-leaf CSV. Near-duplicate augmented
  copies are **not** detected here; the v1 pHash audit still applies before any external claim.
- A crop with a single observed condition gets no disease head and is exported in `single_class_crops`; the
  API now reports that as `status='single_class_crop'`, not as a verified diagnosis.
- The global 38-class head is exported for future fallback use; `main.py` does not read it yet.

## Cell 1 - Imports and GPU

In [ ]:
import os, re, json, time, hashlib, shutil
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageOps
from transformers import AutoModel, CLIPVisionModel
from safetensors import safe_open
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, confusion_matrix
from scipy.optimize import minimize_scalar
import matplotlib.pyplot as plt

assert torch.cuda.is_available(), 'Enable a Kaggle GPU.'
DEVICE = 'cuda'
print(torch.__version__, torch.cuda.get_device_name(0))

## Cell 2 - Paths and protocol

In [ ]:
PV_ROOT = '/kaggle/input/datasets/mohamedmohamed778/pv-images'
PV_GROUP_CSV = None      # optional csv: path (or relative_path) + group_id for shared leaves/sessions
PD_ROOT = '/kaggle/input/datasets/nirmalsankalana/plantdoc-dataset'
ENCODER_BUNDLE = '/kaggle/input/datasets/mohamedmohamed778/i-hope-it-is-one-before-final/multicrop_bundle'
OUT_BASE = Path('/kaggle/working/pv_pd_multicrop_v2')

SEED = 42

# --- feature recipe -------------------------------------------------------
FEATURE_MODE = 'topk_norm'      # 'topk_norm' (v2) or 'mean_patch' (identical to v1 features)
TOPK = 8
USE_ZOOM_VIEW = True            # adds a centered 80% crop view as a third augmented view

# --- data protocol -------------------------------------------------------
MIN_CLASS_TRAIN = 20            # a class enters a head only with this much training data
PD_VAL_FRACTION = .15
PV_CAP_RATIO = 2.0              # at most this many PlantVillage images per PlantDoc field image
FOLDS = 5
PD_MAPPING_OVERRIDES_VERIFIED = False
PD_MAPPING_OVERRIDES = {
    'Apple rust leaf': 'Apple___Cedar_apple_rust',
    'Bell pepper leaf spot': 'Pepper,_bell___Bacterial_spot',
    'Corn leaf blight': 'Corn_(maize)___Northern_Leaf_Blight',
}

# --- head training -------------------------------------------------------
HEAD_CFG = dict(in_drop=.2, hidden=1024, drop2=.3)
EPOCHS = 40
PATIENCE = 8
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-2
LABEL_SMOOTHING = .05
TRAIN_BATCH_SIZE = 256
EXTRACT_BATCH_SIZE = 16
NUM_WORKERS = 2

FEATURE_DIM = 2816
CLIP_NAME = 'openai/clip-vit-large-patch14-336'
DINO_NAME = 'facebook/dinov2-large'
RECIPE = dict(
    clip_model=CLIP_NAME, dino_model=DINO_NAME, clip_size=336, dino_size=224,
    resize='PIL bicubic direct resize', views='original, hflip and centered 80% crop; mean classifier logits',
    clip_mean=[0.48145466, 0.4578275, 0.40821073], clip_std=[0.26862954, 0.26130258, 0.27577711],
    dino_mean=[0.485, 0.456, 0.406], dino_std=[0.229, 0.224, 0.225],
    features='normalized CLIP image embedding, DINO CLS, then ' + FEATURE_MODE + ' DINO patch tokens',
    patch_pool=FEATURE_MODE, topk=TOPK, version='disease_heads_v2',
)
OUT_BASE.mkdir(parents=True, exist_ok=True)
ROOT = OUT_BASE / ('run_' + time.strftime('%Y%m%d_%H%M%S'))
ROOT.mkdir(parents=True, exist_ok=False)
CACHE = OUT_BASE / 'feature_cache'
CACHE.mkdir(exist_ok=True)
for needed in (PV_ROOT, PD_ROOT, ENCODER_BUNDLE):
    assert Path(needed).exists(), 'Missing input: ' + str(needed)
print('run dir:', ROOT)

## Cell 3 - Full label inventory (this is the fix)

In [ ]:
# The 38 PlantVillage classes, in the order the app and the previous heads used.
PV_CLASSES = [
    'Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy',
    'Blueberry___healthy',
    'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy',
    'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_',
    'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy',
    'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)',
    'Grape___healthy',
    'Orange___Haunglongbing_(Citrus_greening)',
    'Peach___Bacterial_spot', 'Peach___healthy',
    'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy',
    'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy',
    'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew',
    'Strawberry___Leaf_scorch', 'Strawberry___healthy',
    'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold',
    'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites Two-spotted_spider_mite', 'Tomato___Target_Spot',
    'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Tomato___Tomato_mosaic_virus', 'Tomato___healthy',
]
TOK_TO_CLASS = {re.sub(r'[^a-z0-9]', '', c.lower()): c for c in PV_CLASSES}
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}


def images_in(root):
    return sorted(p for p in Path(root).rglob('*') if p.is_file() and p.suffix.lower() in IMG_EXT)


def canonical(label):
    return TOK_TO_CLASS.get(re.sub(r'[^a-z0-9]', '', str(label).lower()))


def crop_of(label):
    return label.split('___', 1)[0]


# PlantDoc folder -> canonical PlantVillage class. None = deliberately unresolved.
PD_BASE_MAPPING = {
    'Apple Scab Leaf': 'Apple___Apple_scab',
    'Apple leaf': 'Apple___healthy',
    'Apple rust leaf': None,
    'Bell pepper leaf': 'Pepper,_bell___healthy',
    'Bell pepper leaf spot': None,
    'Blueberry leaf': 'Blueberry___healthy',
    'Cherry leaf': 'Cherry_(including_sour)___healthy',
    'Corn Gray leaf spot': 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot',
    'Corn rust leaf': 'Corn_(maize)___Common_rust_',
    'Corn leaf blight': None,
    'Peach leaf': 'Peach___healthy',
    'Potato leaf': 'Potato___healthy',
    'Potato leaf early blight': 'Potato___Early_blight',
    'Potato leaf late blight': 'Potato___Late_blight',
    'Raspberry leaf': 'Raspberry___healthy',
    'Soyabean leaf': 'Soybean___healthy',
    'Soybean leaf': 'Soybean___healthy',
    'Squash Powdery mildew leaf': 'Squash___Powdery_mildew',
    'Strawberry leaf': 'Strawberry___healthy',
    'grape leaf': 'Grape___healthy',
    'grape leaf black rot': 'Grape___Black_rot',
    'Tomato Early blight leaf': 'Tomato___Early_blight',
    'Tomato leaf late blight': 'Tomato___Late_blight',
    'Tomato leaf': 'Tomato___healthy',
}
if PD_MAPPING_OVERRIDES and not PD_MAPPING_OVERRIDES_VERIFIED:
    raise RuntimeError(
        'Check the PlantDoc folders in PD_MAPPING_OVERRIDES by eye, then set '
        'PD_MAPPING_OVERRIDES_VERIFIED = True (or clear PD_MAPPING_OVERRIDES).'
    )
PD_MAPPING = {k: v for k, v in {**PD_BASE_MAPPING, **PD_MAPPING_OVERRIDES}.items() if v}

pv_rows, pd_rows, unresolved = [], [], []

for folder in sorted(p for p in Path(PV_ROOT).iterdir() if p.is_dir()):
    files = images_in(folder)
    label = canonical(folder.name)
    if label is None:
        if files:
            unresolved.append(dict(source='PlantVillage', folder=folder.name, images=len(files), reason='not a PlantVillage class'))
        continue
    pv_rows += [dict(source='PlantVillage', path=str(f), label=label) for f in files]

for split in ('train', 'test'):
    split_root = Path(PD_ROOT) / split
    assert split_root.is_dir(), 'PlantDoc split missing: ' + str(split_root)
    for folder in sorted(p for p in split_root.iterdir() if p.is_dir()):
        files = images_in(folder)
        label = PD_MAPPING.get(folder.name)
        if label is None:
            if files:
                unresolved.append(dict(source='PlantDoc/' + split, folder=folder.name, images=len(files), reason='unmapped or unverified'))
            continue
        pd_rows += [dict(source='PlantDoc', path=str(f), label=label, split=split) for f in files]

PV_FRAME_RAW = pd.DataFrame(pv_rows)
PD_FRAME_RAW = pd.DataFrame(pd_rows)
assert len(PV_FRAME_RAW) and len(PD_FRAME_RAW), 'No images found - check PV_ROOT / PD_ROOT.'
UNRESOLVED = pd.DataFrame(unresolved)
UNRESOLVED.to_csv(ROOT / 'unresolved_folders.csv', index=False)

COUNTS = pd.DataFrame(0, index=PV_CLASSES, columns=['PlantVillage', 'PlantDoc'])
for label, n in PV_FRAME_RAW.label.value_counts().items():
    COUNTS.loc[label, 'PlantVillage'] = int(n)
for label, n in PD_FRAME_RAW.label.value_counts().items():
    COUNTS.loc[label, 'PlantDoc'] = int(n)
COUNTS['crop'] = [crop_of(l) for l in COUNTS.index]
COUNTS['total'] = COUNTS.PlantVillage + COUNTS.PlantDoc
COUNTS['usable'] = COUNTS.total >= MIN_CLASS_TRAIN
COUNTS.to_csv(ROOT / 'label_inventory.csv')

LABELS_BY_CROP = defaultdict(list)
for label in COUNTS.index[COUNTS.usable]:
    LABELS_BY_CROP[crop_of(label)].append(label)
SINGLE_CLASS = {c: v[0] for c, v in LABELS_BY_CROP.items() if len(v) == 1}
DISEASE_CROPS = {c: v for c, v in LABELS_BY_CROP.items() if len(v) >= 2}

print('classes in inventory:', int(COUNTS.usable.sum()), 'of', len(PV_CLASSES))
print('crops with a disease head:', len(DISEASE_CROPS), sorted(DISEASE_CROPS))
print('single-class crops:', SINGLE_CLASS)
print('PD images usable:', int(PD_FRAME_RAW.label.isin(COUNTS.index[COUNTS.usable]).sum()), 'of', len(PD_FRAME_RAW))
display(COUNTS)
if len(UNRESOLVED):
    print('unresolved folders (not trained on):')
    display(UNRESOLVED)

## Cell 4 - Frozen encoders and cached features

Feature layout is unchanged at 2816 dims: `CLIP projection (768) + DINO CLS (2048) + pooled DINO patches (2048)`.
Only the patch pooling changes (`topk_norm`), so head checkpoints keep the v1 shape and `main.py` keeps working.

In [ ]:
def sha_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


def load_local(cls, path):
    try:
        model = cls.from_pretrained(str(path), local_files_only=True, dtype=torch.float16)
    except TypeError:
        model = cls.from_pretrained(str(path), local_files_only=True, torch_dtype=torch.float16)
    return model.to(DEVICE).eval()


ENC = Path(ENCODER_BUNDLE)
clip_dirs = sorted(p for p in ENC.rglob('config.json') if p.parent.name == 'clip')
dino_dirs = sorted(p for p in ENC.rglob('config.json') if p.parent.name == 'dino')
assert clip_dirs and dino_dirs, 'ENCODER_BUNDLE must contain clip/ and dino/ with config.json'
CLIP_DIR, DINO_DIR = clip_dirs[0].parent, dino_dirs[0].parent

clip_model = load_local(CLIPVisionModel, CLIP_DIR)
clip_model.requires_grad_(False)
clip_weights = sorted(CLIP_DIR.glob('*.safetensors'))
assert clip_weights, 'clip/model.safetensors missing'
with safe_open(str(clip_weights[0]), framework='pt') as f:
    clip_proj = f.get_tensor('visual_projection.weight').to(DEVICE).half()

dino_model = load_local(AutoModel, DINO_DIR)
dino_model.requires_grad_(False)
print('encoders loaded from', CLIP_DIR.parent)


class ViewBank(Dataset):
    def __init__(self, paths, size, zoom):
        self.paths, self.size, self.zoom = list(paths), size, zoom

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        with Image.open(self.paths[i]) as im:
            rgb = ImageOps.exif_transpose(im).convert('RGB')
        base = rgb.resize((self.size, self.size), Image.Resampling.BICUBIC)
        views = [base, base.transpose(Image.Transpose.FLIP_LEFT_RIGHT)]
        if self.zoom:
            w, h = rgb.size
            s = int(.8 * min(w, h))
            cx, cy = w // 2, h // 2
            box = (max(0, cx - s // 2), max(0, cy - s // 2), min(w, cx + s // 2), min(h, cy + s // 2))
            if box[2] - box[0] >= 16 and box[3] - box[1] >= 16:
                views.append(rgb.crop(box).resize((self.size, self.size), Image.Resampling.BICUBIC))
            else:
                views.append(base)
        return torch.from_numpy(np.stack([np.asarray(v) for v in views]).copy()).permute(0, 3, 1, 2).float()


NVIEWS = 3 if USE_ZOOM_VIEW else 2


def cache_path(kind, frame):
    key = hashlib.sha256(json.dumps([RECIPE, kind, NVIEWS, frame.path.tolist()]).encode()).hexdigest()[:24]
    return CACHE / (kind + '_' + key + '.npy')


@torch.inference_mode()
def extract(kind, frame):
    target = cache_path(kind, frame)
    if target.exists():
        return np.load(target, mmap_mode='r')
    size = int(RECIPE[kind + '_size'])
    dim = 768 if kind == 'clip' else 2048
    out = np.lib.format.open_memmap(target, mode='w+', dtype='float32', shape=(len(frame), NVIEWS, dim))
    mean = torch.tensor(RECIPE[kind + '_mean'], device=DEVICE).half().view(1, 1, 3, 1, 1)
    std = torch.tensor(RECIPE[kind + '_std'], device=DEVICE).half().view(1, 1, 3, 1, 1)
    loader = DataLoader(ViewBank(frame.path.tolist(), size, USE_ZOOM_VIEW),
                        batch_size=EXTRACT_BATCH_SIZE, num_workers=NUM_WORKERS)
    offset = 0
    for step, batch in enumerate(loader):
        x = ((batch.to(DEVICE, non_blocking=True).half() / 255) - mean) / std
        b, v = x.shape[0], x.shape[1]
        flat = x.flatten(0, 1)
        if kind == 'clip':
            pooled = clip_model(pixel_values=flat).pooler_output
            feats = F.normalize(F.linear(pooled, clip_proj).float(), dim=-1)
        else:
            hidden = dino_model(pixel_values=flat).last_hidden_state
            cls, patches = hidden[:, 0], hidden[:, 1:]
            if FEATURE_MODE == 'topk_norm':
                k = min(TOPK, patches.shape[1])
                idx = patches.norm(dim=-1).topk(k, dim=-1).indices
                pooled_patch = patches.gather(1, idx[..., None].expand(-1, -1, patches.shape[-1])).mean(1)
            else:
                pooled_patch = patches.mean(1)
            feats = torch.cat([cls, pooled_patch], dim=-1).float()
        out[offset:offset + b] = feats.view(b, v, dim).cpu().numpy()
        offset += b
        if step % 100 == 0:
            print(kind, offset, '/', len(frame), flush=True)
    out.flush()
    return out


def features_for(frame):
    clip_feat = np.asarray(extract('clip', frame))
    dino_feat = np.asarray(extract('dino', frame))
    return np.concatenate([clip_feat, dino_feat], axis=-1).astype(np.float32)


print('NVIEWS =', NVIEWS, '| feature dim =', FEATURE_DIM)

## Cell 5 - Duplicate groups and the PlantDoc split

In [ ]:
PV_FRAME = PV_FRAME_RAW.copy()
PD_FRAME = PD_FRAME_RAW.copy()
for frame in (PV_FRAME, PD_FRAME):
    frame['sha'] = [sha_file(p) for p in frame.path]
    frame['group'] = frame.sha

if PV_GROUP_CSV:
    manual = pd.read_csv(PV_GROUP_CSV)
    col = 'relative_path' if 'relative_path' in manual.columns else 'path'
    manual['sha'] = [sha_file(p) for p in manual[col]]
    lookup = dict(zip(manual.sha, manual.group_id.astype(str)))
    for frame in (PV_FRAME, PD_FRAME):
        frame['group'] = [lookup.get(s, g) for s, g in zip(frame.sha, frame.group)]

PD_TRAINPOOL = PD_FRAME[PD_FRAME.split == 'train'].reset_index(drop=True)
PD_TEST = PD_FRAME[PD_FRAME.split == 'test'].reset_index(drop=True)

val_folds = max(2, int(round(1 / PD_VAL_FRACTION)))
pd_tr_idx, pd_va_idx = next(StratifiedGroupKFold(
    n_splits=val_folds, shuffle=True, random_state=SEED
).split(PD_TRAINPOOL, PD_TRAINPOOL.label, PD_TRAINPOOL.group))
PD_TRAIN = PD_TRAINPOOL.iloc[pd_tr_idx].reset_index(drop=True)
PD_VAL = PD_TRAINPOOL.iloc[pd_va_idx].reset_index(drop=True)

USABLE = set(COUNTS.index[COUNTS.usable])
TRAIN_FRAME = pd.concat([
    PV_FRAME[PV_FRAME.label.isin(USABLE)],
    PD_TRAIN[PD_TRAIN.label.isin(USABLE)],
], ignore_index=True)
TRAIN_FRAME['crop'] = [crop_of(l) for l in TRAIN_FRAME.label]
PD_VAL = PD_VAL[PD_VAL.label.isin(USABLE)].reset_index(drop=True)
PD_TEST = PD_TEST[PD_TEST.label.isin(USABLE)].reset_index(drop=True)

print('train rows:', len(TRAIN_FRAME), '| PlantDoc val:', len(PD_VAL), '| PlantDoc test:', len(PD_TEST))
print(TRAIN_FRAME.groupby('source').size().to_dict())


def capped_frame(labels, frame=None):
    frame = TRAIN_FRAME if frame is None else frame
    parts = []
    for label in labels:
        part = frame[frame.label == label]
        pv_part = part[part.source == 'PlantVillage']
        pd_part = part[part.source != 'PlantVillage']
        cap = max(MIN_CLASS_TRAIN, int(round(PV_CAP_RATIO * len(pd_part))))
        if len(pv_part) > cap:
            pv_part = pv_part.sample(cap, random_state=SEED)
        parts.append(pd.concat([pv_part, pd_part]))
    return pd.concat(parts, ignore_index=True)

## Cell 6 - Head training with grouped 5-fold CV

In [ ]:
class ArrayData(Dataset):
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.x[i], int(self.y[i])


def make_head(dim, k, cfg):
    if cfg['hidden'] == 0:
        return nn.Sequential(nn.Dropout(cfg['in_drop']), nn.Linear(dim, k))
    return nn.Sequential(
        nn.Dropout(cfg['in_drop']), nn.Linear(dim, cfg['hidden']), nn.GELU(),
        nn.Dropout(cfg['drop2']), nn.Linear(cfg['hidden'], k),
    )


@torch.inference_mode()
def predict_proba(head, mu, sd, x):
    head.eval()
    out = []
    mu_t = torch.from_numpy(mu).to(DEVICE).view(1, 1, -1)
    sd_t = torch.from_numpy(sd).to(DEVICE).view(1, 1, -1)
    for start in range(0, len(x), 512):
        xb = torch.from_numpy(np.asarray(x[start:start + 512])).to(DEVICE)
        logits = head((xb - mu_t) / sd_t).mean(1)
        out.append(torch.softmax(logits, dim=-1).cpu().numpy())
    return np.concatenate(out) if out else np.empty((0, 0), np.float32)


def fit_head(x, y, epochs, val=None, seed=SEED):
    torch.manual_seed(seed)
    np.random.seed(seed)
    mu = x.mean(axis=(0, 1), dtype=np.float64).astype(np.float32)
    sd = x.std(axis=(0, 1), dtype=np.float64).astype(np.float32)
    sd[sd < 1e-6] = 1e-6
    k = int(y.max()) + 1
    head = make_head(FEATURE_DIM, k, HEAD_CFG).to(DEVICE)
    opt = torch.optim.AdamW(head.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, epochs))
    lossf = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    loader = DataLoader(ArrayData(x, y), batch_size=TRAIN_BATCH_SIZE, shuffle=True,
                        num_workers=0, drop_last=len(y) > TRAIN_BATCH_SIZE)
    mu_t = torch.from_numpy(mu).to(DEVICE).view(1, 1, -1)
    sd_t = torch.from_numpy(sd).to(DEVICE).view(1, 1, -1)
    best = dict(f1=-1.0, epoch=epochs, state=None)
    stale = 0
    for epoch in range(1, epochs + 1):
        head.train()
        for xb, yb in loader:
            logits = head((xb.to(DEVICE) - mu_t) / sd_t).mean(1)
            loss = lossf(logits, yb.to(DEVICE))
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
        sched.step()
        if val is not None:
            prob = predict_proba(head, mu, sd, val[0])
            score = f1_score(val[1], prob.argmax(1), average='macro', zero_division=0)
            if score > best['f1'] + 1e-4:
                best = dict(f1=float(score), epoch=epoch,
                            state={k_: v_.detach().clone() for k_, v_ in head.state_dict().items()})
                stale = 0
            else:
                stale += 1
                if stale >= PATIENCE:
                    break
    if best['state'] is not None:
        head.load_state_dict(best['state'])
    return head, mu, sd, best


def train_head_cv(frame, labels, tag, epochs=EPOCHS):
    labels = sorted(labels)
    class_to_idx = {label: i for i, label in enumerate(labels)}
    frame = frame[frame.label.isin(labels)].reset_index(drop=True)
    x = features_for(frame)
    y = frame.label.map(class_to_idx).to_numpy()
    groups = frame.group.to_numpy()

    n_splits = min(FOLDS, len(np.unique(groups)), int(np.bincount(y).min()))
    oof = np.zeros((len(y), len(labels)), np.float32)
    oof_mask = np.zeros(len(y), bool)
    fold_f1, best_epochs = [], []

    if n_splits >= 2:
        splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
        for fold, (tr, va) in enumerate(splitter.split(np.zeros(len(y)), y, groups)):
            head, mu, sd, best = fit_head(np.asarray(x[tr]), y[tr], epochs, val=(np.asarray(x[va]), y[va]))
            oof[va] = predict_proba(head, mu, sd, np.asarray(x[va]))
            oof_mask[va] = True
            fold_f1.append(best['f1'])
            best_epochs.append(max(1, best['epoch']))
            print('  ' + tag + ' fold ' + str(fold) + ' f1=' + format(best['f1'], '.4f') + ' epoch=' + str(best['epoch']), flush=True)
            del head
            torch.cuda.empty_cache()
        oof_f1 = float(f1_score(y[oof_mask], oof[oof_mask].argmax(1), average='macro', zero_division=0))
        selected_epoch = int(np.median(best_epochs))
    else:
        # Too few groups for CV: one hold-out, chosen epochs fixed to EPOCHS.
        rng = np.random.default_rng(SEED)
        order = rng.permutation(len(y))
        cut = max(1, int(.2 * len(y)))
        va, tr = order[:cut], order[cut:]
        head, mu, sd, best = fit_head(np.asarray(x[tr]), y[tr], epochs, val=(np.asarray(x[va]), y[va]))
        oof[va] = predict_proba(head, mu, sd, np.asarray(x[va]))
        oof_mask[va] = True
        oof_f1 = float(f1_score(y[va], oof[va].argmax(1), average='macro', zero_division=0))
        selected_epoch = max(1, best['epoch'])
        fold_f1, best_epochs = [best['f1']], [selected_epoch]
        del head
        torch.cuda.empty_cache()

    print(tag + ': OOF macro F1=' + format(oof_f1, '.4f') + ' selected_epoch=' + str(selected_epoch), flush=True)
    head, mu, sd, _ = fit_head(np.asarray(x), y, selected_epoch)
    checkpoint = dict(
        state={k_: v_.detach().cpu() for k_, v_ in head.state_dict().items()},
        cfg=HEAD_CFG, class_to_idx=class_to_idx,
        mu=torch.from_numpy(mu), sd=torch.from_numpy(sd),
        recipe=RECIPE, feature_dim=FEATURE_DIM, tag=tag,
        oof_macro_f1=oof_f1, selected_epoch=selected_epoch,
        cv=dict(folds=len(fold_f1), fold_f1=[float(v) for v in fold_f1], best_epochs=best_epochs),
    )
    del head
    torch.cuda.empty_cache()
    return checkpoint

## Cell 7 - Crop router and per-crop disease heads

In [ ]:
ROUTER_FRAME = TRAIN_FRAME.assign(label=TRAIN_FRAME.crop)   # the router predicts the crop itself
print('=== crop router ===')
CROP_ROUTER = train_head_cv(ROUTER_FRAME, sorted(TRAIN_FRAME.crop.unique()), 'crop_router')

HEADS = {}
for crop, labels in sorted(DISEASE_CROPS.items()):
    frame = capped_frame(labels)
    usable = [l for l in labels if (frame.label == l).sum() >= MIN_CLASS_TRAIN and (frame.label == l).sum() >= 2]
    if len(usable) < 2:
        SINGLE_CLASS[crop] = usable[0] if usable else labels[0]
        print(crop + ': fewer than two usable conditions -> single class ' + SINGLE_CLASS[crop])
        continue
    print('=== ' + crop + ' (' + str(len(usable)) + ' classes) ===')
    HEADS[crop] = train_head_cv(frame[frame.label.isin(usable)], usable, crop)

print('heads trained:', sorted(HEADS))
print('single-class crops:', SINGLE_CLASS)

## Cell 8 - Global 38-class fallback head

A single coarse head over every class in the inventory. It is exported as `global_head.pt` so a future
serving change can answer for crops without their own head instead of saying nothing. The current
`main.py` ignores it.

In [ ]:
GLOBAL_LABELS = sorted(COUNTS.index[COUNTS.usable])
GLOBAL_HEAD = train_head_cv(TRAIN_FRAME, GLOBAL_LABELS, 'global_38', epochs=max(10, EPOCHS // 2))

## Cell 9 - Calibration and acceptance thresholds

In [ ]:
def raw_logits(checkpoint, x):
    head = make_head(FEATURE_DIM, len(checkpoint['class_to_idx']), checkpoint['cfg']).to(DEVICE)
    head.load_state_dict({k: v.to(DEVICE) for k, v in checkpoint['state'].items()})
    head.eval()
    mu = checkpoint['mu'].numpy()
    sd = checkpoint['sd'].numpy()
    prob = predict_proba(head, mu, sd, x)
    del head
    torch.cuda.empty_cache()
    return np.log(np.clip(prob, 1e-12, 1.0)).astype(np.float64)


def calibrate(checkpoint, frame):
    idx = checkpoint['class_to_idx']
    frame = frame[frame.label.isin(idx)]
    if len(frame) < 10:
        checkpoint['temperature'] = 1.0
        checkpoint['threshold'] = 0.0
        checkpoint['calibration_n'] = int(len(frame))
        return checkpoint
    x = features_for(frame)
    logits = raw_logits(checkpoint, x)
    y = frame.label.map(idx).to_numpy()

    def nll(log_t):
        t = float(np.exp(log_t))
        z = logits / t
        z = z - z.max(1, keepdims=True)
        logp = z - np.log(np.exp(z).sum(1, keepdims=True))
        return -logp[np.arange(len(y)), y].mean()

    t = float(np.exp(minimize_scalar(nll, bounds=(-1.0, 1.6), method='bounded').x))
    prob = np.exp((logits / t) - (logits / t).max(1, keepdims=True))
    prob /= prob.sum(1, keepdims=True)
    conf = prob.max(1)
    correct = prob.argmax(1) == y
    best = dict(threshold=0.0, coverage=1.0, accuracy=float(correct.mean()))
    for thr in np.arange(.0, 1.0, .05):
        keep = conf >= thr
        if keep.sum() >= max(10, .2 * len(y)):
            acc = float(correct[keep].mean())
            if acc >= .90 and float(keep.mean()) > best['coverage']:
                best = dict(threshold=float(round(thr, 2)), coverage=float(keep.mean()), accuracy=acc)
    checkpoint.update(temperature=t, calibration_n=int(len(y)), **{
        'threshold': best['threshold'], 'accepted_coverage': best['coverage'],
        'accepted_accuracy': best['accuracy'],
    })
    return checkpoint


for crop in list(HEADS):
    frame = PD_VAL[PD_VAL.label.isin(HEADS[crop]['class_to_idx'])]
    HEADS[crop] = calibrate(HEADS[crop], frame)
    ck = HEADS[crop]
    print(crop + ': T=' + format(ck['temperature'], '.3f') + ' threshold=' + str(ck['threshold']) +
          ' PD-val coverage=' + format(ck['accepted_coverage'], '.2f') + ' acc=' + format(ck['accepted_accuracy'], '.3f'))
GLOBAL_HEAD = calibrate(GLOBAL_HEAD, PD_VAL)
CROP_ROUTER = calibrate(CROP_ROUTER, PD_VAL)

## Cell 10 - Export the v2 bundle

In [ ]:
def slug(name):
    return re.sub(r'[^A-Za-z0-9]+', '_', name).strip('_')


BUNDLE = ROOT / 'bundle'
(BUNDLE / 'heads').mkdir(parents=True)

for crop, checkpoint in HEADS.items():
    torch.save(checkpoint, BUNDLE / 'heads' / (slug(crop) + '.pt'))
torch.save(CROP_ROUTER, BUNDLE / 'crop_head.pt')
torch.save(GLOBAL_HEAD, BUNDLE / 'global_head.pt')

encoder_identity = hashlib.sha256(json.dumps(
    [(str(p.relative_to(ENC)), sha_file(p)) for p in sorted(ENC.rglob('*'))
     if p.is_file() and p.parent.name in ('clip', 'dino') and p.suffix in ('.safetensors', '.json')],
    sort_keys=True).encode()).hexdigest()

crops = sorted(CROP_ROUTER['class_to_idx'])
config = {
    'task': 'pv_pd_multicrop_expanded_tomato_v2',
    'recipe': RECIPE,
    'feature_dim': FEATURE_DIM,
    'feature_mode': FEATURE_MODE,
    'topk': TOPK,
    'encoder_identity': encoder_identity,
    'crop_class_to_idx': {c: i for i, c in enumerate(crops)},
    'disease_heads': {crop: 'heads/' + slug(crop) + '.pt' for crop in sorted(HEADS)},
    'single_class_crops': SINGLE_CLASS,
    'global_head': 'global_head.pt',
    'global_classes': {label: i for i, label in enumerate(GLOBAL_LABELS)},
    'protocol': {
        'seed': SEED,
        'selection': 'grouped ' + str(FOLDS) + '-fold CV out-of-fold macro F1 (PlantDoc validation reported only)',
        'min_class_train': MIN_CLASS_TRAIN,
        'pd_val_fraction': PD_VAL_FRACTION,
        'pv_cap_ratio': PV_CAP_RATIO,
        'feature_mode': FEATURE_MODE,
        'topk': TOPK,
        'views': RECIPE['views'],
        'label_inventory': {label: int(COUNTS.loc[label, 'total']) for label in COUNTS.index[COUNTS.usable]},
        'pd_mapping': PD_MAPPING,
        'heads': {crop: {k: v for k, v in HEADS[crop].items() if k in ('oof_macro_f1', 'selected_epoch', 'cv', 'temperature', 'threshold', 'accepted_coverage', 'accepted_accuracy')}
                  for crop in HEADS},
        'crop_router': {k: v for k, v in CROP_ROUTER.items() if k in ('oof_macro_f1', 'selected_epoch', 'cv')},
        'global_head': {k: v for k, v in GLOBAL_HEAD.items() if k in ('oof_macro_f1', 'selected_epoch', 'cv')},
        'unresolved_folders': UNRESOLVED.to_dict('records'),
    },
}
(BUNDLE / 'config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')

manifest = {}
for path in sorted(BUNDLE.rglob('*')):
    if path.is_file():
        manifest[str(path.relative_to(BUNDLE))] = sha_file(path)
(BUNDLE / 'sha256.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')

zip_path = shutil.make_archive(str(ROOT / 'pv_pd_multicrop_v2_bundle'), 'zip', BUNDLE)
print('bundle:', zip_path, format(Path(zip_path).stat().st_size / 1e6, '.1f'), 'MB')
print('heads:', len(HEADS), '| classes per head:', {c: len(h['class_to_idx']) for c, h in HEADS.items()})

## Cell 11 - Report

In [ ]:
def load_head(checkpoint):
    head = make_head(FEATURE_DIM, len(checkpoint['class_to_idx']), checkpoint['cfg']).to(DEVICE)
    head.load_state_dict({k: v.to(DEVICE) for k, v in checkpoint['state'].items()})
    return head.eval()


rows = []
for crop, checkpoint in HEADS.items():
    frame = PD_VAL[PD_VAL.label.isin(checkpoint['class_to_idx'])]
    if len(frame):
        head = load_head(checkpoint)
        prob = predict_proba(head, checkpoint['mu'].numpy(), checkpoint['sd'].numpy(), features_for(frame))
        pd_f1 = float(f1_score(frame.label.map(checkpoint['class_to_idx']), prob.argmax(1), average='macro', zero_division=0))
        del head
        torch.cuda.empty_cache()
    else:
        pd_f1 = float('nan')
    rows.append(dict(crop=crop, classes=len(checkpoint['class_to_idx']), oof_macro_f1=checkpoint['oof_macro_f1'],
                     pd_val_macro_f1=pd_f1, selected_epoch=checkpoint['selected_epoch'],
                     temperature=checkpoint['temperature'], threshold=checkpoint['threshold']))

REPORT = pd.DataFrame(rows).sort_values('oof_macro_f1')
REPORT.to_csv(ROOT / 'heads_report.csv', index=False)
display(REPORT)

if len(PD_TEST):
    worst = REPORT.head(3).crop.tolist()
    fig, axes = plt.subplots(1, len(worst), figsize=(5 * len(worst), 4))
    axes = np.atleast_1d(axes)
    for ax, crop in zip(axes, worst):
        ck = HEADS[crop]
        frame = PD_TEST[PD_TEST.label.isin(ck['class_to_idx'])]
        if not len(frame):
            ax.set_axis_off()
            continue
        head = load_head(ck)
        prob = predict_proba(head, ck['mu'].numpy(), ck['sd'].numpy(), features_for(frame))
        cm = confusion_matrix(frame.label.map(ck['class_to_idx']), prob.argmax(1),
                              labels=list(range(len(ck['class_to_idx']))), normalize='true')
        ax.imshow(cm, cmap='Greens', vmin=0, vmax=1)
        ax.set_title(crop + ' (PlantDoc test)')
        ax.set_xlabel('predicted')
        ax.set_ylabel('true')
    plt.tight_layout()
    plt.savefig(ROOT / 'pd_test_confusions.png', dpi=130)
    plt.show()

print('artifacts in', ROOT)